<a href="https://colab.research.google.com/github/Tshoki-m/Capstone-Research/blob/main/Copy_of_MNIST_PEPITA_TenSeed_Continuation_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST PEPITA: continue the repository-based ten-seed runs

Run this after the current repository MNIST notebook finishes and downloads `mnist_repo_pepita_runs.zip`. That notebook requested seed 0, but the repository treats zero as “no seed supplied” and selected random seeds instead. The actual seeds are in each archived `config.yaml`; this continuation reads those values. It runs seeds 1–9 for both settings and combines them with those two recorded seeds, giving ten independent runs per setting. Do not include seed 0 from the separate, self-contained MNIST notebook: it is a different implementation.

Each run is archived to Google Drive as soon as it finishes. If Colab disconnects, reconnect, rerun setup, upload the same project source ZIP and seed-0 archive, and rerun the training cell; completed seeds are skipped. This is a long sequential GPU job.

Preprocessing follows the supplied repository: images are converted to [0,1] tensors without MNIST mean/std normalization. Test-loader metrics are logged each epoch; the final epoch is used without selecting the best test epoch.


## 1. Select GPU, mount Drive, and upload two ZIP files
Upload the complete PEPITA project source ZIP and the `mnist_repo_pepita_runs.zip` downloaded after the seed-0 notebook completes. The cell identifies them by contents.

In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,shutil,statistics,tempfile,re
from datetime import datetime,timezone
from google.colab import drive,files
print(sys.version)
!nvidia-smi
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/PEPITA_ten_seed_runs/mnist'); ROOT.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
assert len(uploaded)==2, 'Upload the complete project ZIP and the seed-0 MNIST run-results ZIP.'
source_zip=None; baseline_zip=None
for name in uploaded:
 with zipfile.ZipFile(name) as z:
  names=z.namelist()
  if any(n.endswith('scripts/train.py') for n in names): source_zip=name
  if any('mnist_repo_pepita_1hidden_noWD_seed0' in n for n in names): baseline_zip=name
assert source_zip and baseline_zip, 'Could not identify both the source ZIP and the MNIST seed-0 results ZIP.'
ex=Path('/content/pepita_src'); shutil.rmtree(ex,ignore_errors=True); ex.mkdir()
with zipfile.ZipFile(source_zip) as z: z.extractall(ex)
ts=list(ex.rglob('scripts/train.py')); assert len(ts)==1
PROJECT=ts[0].parent.parent.resolve(); assert (PROJECT/'pepita/core/trainer.py').exists(); os.chdir(PROJECT)
print('Project:',PROJECT,'seed-0 archive:',baseline_zip)


3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Sat Oct 31 21:28:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A

## 2. Install the pinned CUDA environment

In [ ]:
os.environ['MAMBA_ROOT_PREFIX']='/content/micromamba'
if not Path('/content/bin/micromamba').exists(): subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba",shell=True,check=True)
MM='/content/bin/micromamba'
subprocess.run([MM,'create','-y','-n','pepita','python=3.10','pip'],check=True)
subprocess.run([MM,'run','-n','pepita','python','-m','pip','install','--upgrade','pip==24.0'],check=True)
pkgs=['torch==1.13.1+cu117','torchvision==0.14.1+cu117','pytorch-lightning==1.6.5','torchmetrics==0.9.2','numpy==1.21.2','Pillow==8.4.0','scikit-learn==1.2.1','scipy==1.10.1','loguru==0.6.0','setuptools==80.9.0','ray[tune]==2.3.1','yacs==0.1.8','tensorboard==2.12.3','protobuf<4']
subprocess.run([MM,'run','-n','pepita','python','-m','pip','install','--extra-index-url','https://download.pytorch.org/whl/cu117',*pkgs],check=True)


## 3. Apply the guarded zero-split fix and check GPU compatibility

In [ ]:
patch=r'''from pathlib import Path
p=Path('pepita/dataset/getdataloader.py'); s=p.read_text(encoding='utf-8')
a='    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True)\n'
b='    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True) if val_data is not None else None\n'
assert s.count(a)==3, f'Expected 3 validation-loader lines, got {s.count(a)}'
p.write_text(s.replace(a,b),encoding='utf-8'); print('Applied guarded val_split=0 loader fix.')
'''
subprocess.run([MM,'run','-n','pepita','python','-c',patch],cwd=PROJECT,check=True)
check="import torch,pytorch_lightning as pl; c=torch.cuda.get_device_capability(0); print(torch.__version__,pl.__version__,torch.cuda.get_device_name(0),c); assert f'sm_{c[0]}{c[1]}' in torch.cuda.get_arch_list()"
subprocess.run([MM,'run','-n','pepita','python','-c',check],check=True)


## 4. Read the completed repository seed-0 metrics
The seed-0 archive is the output of the repository-based notebook, not the independent reimplementation.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def read_scalars_from_zip(zf, experiment):
 files=[n for n in zf.namelist() if experiment in n and 'events.out.tfevents.' in n and not any(x in n for x in ('angle_al_total','weight_norms','b_means','b_stds'))]
 if not files: raise FileNotFoundError(f'No main event file for {experiment}')
 tmp=Path('/content')/(experiment+'_events'); tmp.mkdir(exist_ok=True)
 event_path=tmp/Path(files[0]).name; event_path.write_bytes(zf.read(files[0]))
 acc=EventAccumulator(str(event_path),size_guidance={'scalars':0}); acc.Reload(); out={}
 for tag in ('val_acc','val_loss'):
  e=max(acc.Scalars(tag),key=lambda x:x.step); out[tag]=float(e.value); out[tag+'_step']=int(e.step)
 return out
BASE={}
with zipfile.ZipFile(baseline_zip) as z:
 for label,exp,wd in [('noWD','mnist_repo_pepita_1hidden_noWD_seed0',0.0),('WD1e-5','mnist_repo_pepita_1hidden_WD1e-5_seed0',1e-5)]:
  cfg_name=next(n for n in z.namelist() if n.endswith('/'+exp+'/config.yaml'))
  cfg_text=z.read(cfg_name).decode()
  actual_seed=int(re.search(r'^SEED_VALUE:\s*(\d+)',cfg_text,re.M).group(1))
  BASE[label]={'seed':actual_seed,'weight_decay':wd,'metrics':read_scalars_from_zip(z,exp),'source':'completed repository run; actual seed from config.yaml'}
print(json.dumps(BASE,indent=2))


## 5. Run seeds 1–9 for both settings
The current run settings match the seed-0 notebook: 1,024 hidden units, learning rate 0.1, 100 epochs, batch size 64, dropout 0.1, momentum 0.9, learning-rate drops at epochs 60 and 90, with WD 0 or 1e-5. Each finished run is saved to Drive.

In [ ]:
SEEDS=list(range(1,10))
PRESETS=[('noWD',0.0),('WD1e-5',1e-5)]
runtime_cmd="import torch,torchvision,pytorch_lightning as pl,platform,json; print(json.dumps({'python':platform.python_version(),'pytorch':torch.__version__,'torchvision':torchvision.__version__,'lightning':pl.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}))"
runtime=json.loads(subprocess.check_output([MM,'run','-n','pepita','python','-c',runtime_cmd],cwd=PROJECT,text=True))
def final_metrics(folder):
 ev=[p for p in folder.rglob('events.out.tfevents.*') if not any(x in str(p) for x in ('angle_al_total','weight_norms','b_means','b_stds'))]
 assert ev, f'No main event file in {folder}'
 a=EventAccumulator(str(ev[0]),size_guidance={'scalars':0}); a.Reload(); r={}
 for tag in ('val_acc','val_loss'):
  e=max(a.Scalars(tag),key=lambda x:x.step); r[tag]=float(e.value); r[tag+'_step']=int(e.step)
 return r
for label,wd in PRESETS:
 for seed in SEEDS:
  exp=f'mnist_repo_pepita_1hidden_{label}_seed{seed}'; dest=ROOT/f'{exp}.zip'
  if dest.exists(): print('Skip archived',exp); continue
  cmd=[MM,'run','-n','pepita','python','scripts/train.py','-en',exp,'-d','mnist','-ls','1024','-lr','0.1','-wd',str(wd),'-e','100','-bs','64','-mom','0.9','-dr','0.1','-lrd','0.1','-de','60','90','-li','he_uniform','-bi','uniform','-bstd','0.05','-w','2','-wmlr','0.1','-wmwd','0.5','-s',str(seed)]
  folder=PROJECT/'experiments'/exp; shutil.rmtree(folder,ignore_errors=True); folder.mkdir(parents=True)
  start=datetime.now(timezone.utc).isoformat(); print('START',exp,start,flush=True)
  with open(folder/'colab_console.log','w',encoding='utf-8') as log:
   p=subprocess.Popen(cmd,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
   for line in p.stdout: print(line,end='',flush=True); log.write(line)
   rc=p.wait()
  end=datetime.now(timezone.utc).isoformat()
  if rc: raise RuntimeError(f'{exp} failed; completed Drive archives remain safe.')
  metrics=final_metrics(folder)
  info={'dataset':'MNIST','experiment':exp,'seed':seed,'weight_decay':wd,'metrics':metrics,'runtime':runtime,'started_utc':start,'finished_utc':end,'evaluation_note':'test loader monitored each epoch; report final epoch only.'}
  (folder/'final_metrics.json').write_text(json.dumps(info,indent=2),encoding='utf-8')
  with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
   for f in folder.rglob('*'):
    if f.is_file(): z.write(f,Path('experiments')/exp/f.relative_to(folder))
  print('SAVED',dest,metrics,flush=True)


## 6. Calculate ten-seed statistics per MNIST setting
Each configuration is summarized separately. A configuration counts as ten-seed only after all nine new archives are saved alongside the seed-0 result.

In [ ]:
summary={}
for label,wd in PRESETS:
 rows=[{'seed':0,'weight_decay':wd,'metrics':BASE[label]['metrics'],'source':BASE[label]['source']}]
 for zp in sorted(ROOT.glob(f'mnist_repo_pepita_1hidden_{label}_seed*.zip')):
  with zipfile.ZipFile(zp) as z:
   name=next(n for n in z.namelist() if n.endswith('/final_metrics.json'))
   rows.append(json.loads(z.read(name).decode()))
 by_seed={r['seed']:r for r in rows}; rows=[by_seed[k] for k in sorted(by_seed)]
 acc=[r['metrics']['val_acc'] for r in rows]; loss=[r['metrics']['val_loss'] for r in rows]
 summary[label]={'n':len(rows),'seed_results':[{'seed':r['seed'],'accuracy':r['metrics']['val_acc'],'loss':r['metrics']['val_loss']} for r in rows],
  'accuracy_mean':statistics.mean(acc),'accuracy_sample_sd':statistics.stdev(acc) if len(acc)>1 else None,
  'loss_mean':statistics.mean(loss),'loss_sample_sd':statistics.stdev(loss) if len(loss)>1 else None,
  'paper_accuracy_mean':0.9802 if label=='noWD' else 0.9812,'paper_accuracy_sd':0.0008}
print(json.dumps(summary,indent=2))
summary_path=ROOT/'mnist_ten_seed_summary.json'; summary_path.write_text(json.dumps(summary,indent=2),encoding='utf-8'); files.download(str(summary_path))
